# Member 4 – False-Positive Analysis
## Hack-vok | Kepler Exoplanet Search Results

**Purpose:** Screen candidate objects for evidence that may indicate false-positive scenarios.

This analysis is deliberately a **screening/prioritization layer**, not a replacement for the Kepler catalog disposition and not a claim that a candidate is a confirmed false positive.

## Inputs
- Member 2 candidate prediction output: `memeber2_candidate_predictions.csv`
- Existing Kepler false-positive flags included in that output
- Transit/planetary features and prepared-data outlier indicator

## Main Questions
1. Which candidates already carry catalog false-positive flags?
2. Which candidates show indicators associated with eclipsing binaries?
3. Which candidates show contamination/systematic indicators?
4. Which candidates should be prioritized for human review?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Locate Member 2 output in common project locations
filename = "memeber2_candidate_predictions.csv"
search_paths = [
    Path(filename),
    Path("../") / filename,
    Path("../member 2") / filename,
    Path("../member2") / filename,
    Path("/mnt/data") / filename,
]

INPUT = next((p for p in search_paths if p.exists()), None)
if INPUT is None:
    raise FileNotFoundError(
        f"Could not find {filename}. Checked: " + ", ".join(str(p) for p in search_paths)
    )

df = pd.read_csv(INPUT)

print("Input file:", INPUT)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
# Required fields for the Member 4 analysis
required = [
    "kepid", "kepoi_name", "koi_disposition",
    "predicted_confirmed_probability", "priority",
    "koi_prad", "koi_depth", "koi_period", "koi_model_snr",
    "outlier_flag",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"
]

missing = [c for c in required if c not in df.columns]
print("Missing required columns:", missing)

assert not missing, "The Member 2 output is missing a required field."


## 1. Catalog False-Positive Evidence

The Kepler candidate table contains four false-positive flag fields:

- `koi_fpflag_nt`: not transit-like
- `koi_fpflag_ss`: stellar eclipse
- `koi_fpflag_co`: centroid offset / contamination evidence
- `koi_fpflag_ec`: ephemeris-match contamination evidence

These flags are treated as the strongest evidence in this project because they come from the source catalog's vetting information.


In [ ]:
fp_cols = ["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]

df["catalog_fp_flag_count"] = df[fp_cols].fillna(0).sum(axis=1)

print("Any catalog FP flag:", int((df["catalog_fp_flag_count"] > 0).sum()))
for c in fp_cols:
    print(f"{c}: {int(df[c].fillna(0).sum())}")

catalog_flagged = df[df["catalog_fp_flag_count"] > 0].copy()

display_cols = [
    "kepoi_name", "koi_score", "koi_prad", "koi_depth", "koi_period",
    "koi_model_snr", *fp_cols, "outlier_flag",
    "predicted_confirmed_probability", "priority"
]
print("\nCatalog-flagged candidates:")
print(catalog_flagged[display_cols].to_string(index=False))


## 2. Eclipsing-Binary Indicators

A very large inferred planetary radius and/or a very deep transit can be warning signs of a stellar companion rather than a planet.

These are **screening indicators only**. They are not sufficient by themselves to establish an eclipsing binary.

The catalog stellar-eclipse flag is retained as the strongest direct indicator.


In [ ]:
df["eb_indicator"] = (
    (df["koi_fpflag_ss"] == 1) |
    (df["koi_prad"] >= 10) |
    (df["koi_depth"] >= 10000)
).astype(int)

print("Candidates with EB screening indicator:", int(df["eb_indicator"].sum()))


## 3. Contamination and Systematic Indicators

Two additional groups are retained:

- **Contamination:** centroid-offset and ephemeris-contamination catalog flags.
- **Potential artifacts:** not-transit-like catalog flag, prepared-data outlier flag, and low model SNR.

These indicators identify candidates that deserve inspection; they do not prove an instrumental artifact.


In [ ]:
df["contamination_indicator"] = (
    (df["koi_fpflag_co"] == 1) |
    (df["koi_fpflag_ec"] == 1)
).astype(int)

df["artifact_indicator"] = (
    (df["koi_fpflag_nt"] == 1) |
    (df["outlier_flag"] == 1) |
    (df["koi_model_snr"] < 10)
).astype(int)

print("Contamination indicators:", int(df["contamination_indicator"].sum()))
print("Artifact/systematic indicators:", int(df["artifact_indicator"].sum()))


## 4. False-Positive Screening Score

The score below is intentionally transparent.

- Catalog FP flag: strongest contribution.
- Very large radius / deep transit: eclipsing-binary warning.
- Moderate radius/depth warning: weaker contribution.
- Prepared-data outlier and low SNR: artifact/systematic warning.
- Very short period combined with a large radius: additional warning.

**Important:** `predicted_confirmed_probability` remains Member 2's model output. It is **not** converted into a false-positive probability.


In [ ]:
df["fp_screening_score"] = (
    df["catalog_fp_flag_count"] * 5
    + (df["koi_prad"] >= 10).astype(int) * 3
    + ((df["koi_prad"] >= 4) & (df["koi_prad"] < 10)).astype(int) * 2
    + (df["koi_depth"] >= 10000).astype(int) * 3
    + ((df["koi_depth"] >= 3000) & (df["koi_depth"] < 10000)).astype(int) * 2
    + df["outlier_flag"].fillna(0).astype(int)
    + (df["koi_model_snr"] < 10).astype(int)
    + ((df["koi_period"] < 2) & (df["koi_prad"] >= 4)).astype(int)
)

df["fp_risk"] = pd.cut(
    df["fp_screening_score"],
    bins=[-1, 2, 4, np.inf],
    labels=["Low", "Medium", "High"]
)

print(df["fp_risk"].value_counts().reindex(["Low", "Medium", "High"]).fillna(0))


## 5. Evidence Explanation

Each candidate receives a human-readable evidence field so Member 5 can integrate the result without reverse-engineering the score.


In [ ]:
def evidence_type(row):
    evidence = []

    if row["koi_fpflag_ss"] == 1:
        evidence.append("Stellar eclipse flag")
    if row["koi_fpflag_co"] == 1:
        evidence.append("Centroid-offset flag")
    if row["koi_fpflag_ec"] == 1:
        evidence.append("Ephemeris-contamination flag")
    if row["koi_fpflag_nt"] == 1:
        evidence.append("Not-transit-like flag")

    if row["koi_prad"] >= 10:
        evidence.append("Very large inferred radius")
    elif row["koi_prad"] >= 4:
        evidence.append("Large inferred radius")

    if row["koi_depth"] >= 10000:
        evidence.append("Very deep transit")
    elif row["koi_depth"] >= 3000:
        evidence.append("Deep transit")

    if row["outlier_flag"] == 1:
        evidence.append("Prepared-data outlier")

    if row["koi_model_snr"] < 10:
        evidence.append("Low model SNR")

    return "; ".join(evidence) if evidence else "No strong screening indicator"

df["evidence"] = df.apply(evidence_type, axis=1)


## 6. Visualizations


In [ ]:
risk_counts = (
    df["fp_risk"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
    .fillna(0)
)

plt.figure(figsize=(7,5))
risk_counts.plot(kind="bar")
plt.xlabel("False-positive screening risk")
plt.ylabel("Number of candidates")
plt.title("False-Positive Screening Risk")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

for risk in ["Low", "Medium", "High"]:
    sub = df[df["fp_risk"] == risk]
    plt.scatter(
        sub["koi_prad"],
        sub["koi_depth"],
        label=risk,
        alpha=0.65
    )

plt.axvline(10, linestyle="--", linewidth=1)
plt.axhline(10000, linestyle="--", linewidth=1)
plt.xlabel("Planetary radius (Earth radii)")
plt.ylabel("Transit depth (ppm)")
plt.title("Candidate Radius vs Transit Depth")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
top = df.sort_values(
    ["fp_screening_score", "koi_depth"],
    ascending=[False, False]
).head(15).sort_values("fp_screening_score")

plt.figure(figsize=(9,7))
plt.barh(top["kepoi_name"], top["fp_screening_score"])
plt.xlabel("False-positive screening score")
plt.ylabel("KOI")
plt.title("Top Candidates Requiring False-Positive Review")
plt.tight_layout()
plt.show()


## 7. Statistical Check

For the candidate set, we use Spearman correlation to describe the relationship between radius and transit depth. This is descriptive rather than causal.


In [ ]:
valid = df[["koi_prad", "koi_depth"]].dropna()

rho, p_value = stats.spearmanr(valid["koi_prad"], valid["koi_depth"])

print("Spearman correlation (radius vs transit depth)")
print("rho =", round(rho, 4))
print("p-value =", p_value)


## 8. Final Member 4 Output

The final file contains:

- Member 2 model probability and priority
- Kepler catalog FP flags
- EB / contamination / artifact indicators
- transparent screening score
- screening risk
- human-readable evidence

This table is intended for **Member 5 – Integration & Prioritization Lead**.


In [ ]:
output_cols = [
    "kepid", "kepoi_name", "koi_disposition", "koi_score",
    "predicted_confirmed_probability", "priority",
    "koi_prad", "koi_depth", "koi_period", "koi_model_snr",
    "outlier_flag", *fp_cols, "catalog_fp_flag_count",
    "eb_indicator", "contamination_indicator", "artifact_indicator",
    "fp_screening_score", "fp_risk", "evidence"
]

final_output = df[output_cols].sort_values(
    ["fp_screening_score", "koi_depth"],
    ascending=[False, False]
)

final_output.to_csv("member4_false_positive_screening.csv", index=False)

review_output = final_output[
    final_output["fp_risk"].isin(["High", "Medium"])
]
review_output.to_csv("member4_review_candidates.csv", index=False)

summary = pd.DataFrame({
    "metric": [
        "Total candidates",
        "Any catalog FP flag",
        "Stellar eclipse flag",
        "Centroid offset flag",
        "Ephemeris contamination flag",
        "Not transit-like flag",
        "Prepared-data outlier",
        "High screening risk",
        "Medium screening risk",
        "Low screening risk"
    ],
    "count": [
        len(df),
        int((df["catalog_fp_flag_count"] > 0).sum()),
        int(df["koi_fpflag_ss"].fillna(0).sum()),
        int(df["koi_fpflag_co"].fillna(0).sum()),
        int(df["koi_fpflag_ec"].fillna(0).sum()),
        int(df["koi_fpflag_nt"].fillna(0).sum()),
        int(df["outlier_flag"].fillna(0).sum()),
        int((df["fp_risk"] == "High").sum()),
        int((df["fp_risk"] == "Medium").sum()),
        int((df["fp_risk"] == "Low").sum())
    ]
})

summary.to_csv("member4_summary.csv", index=False)

print(summary.to_string(index=False))


## 9. Interpretation for the Report

- The candidate set contains 438 objects.
- Three candidates carry the catalog stellar-eclipse flag.
- Large inferred radius and deep transit are useful screening indicators for possible stellar eclipses, but they are not proof of an eclipsing binary.
- Prepared-data outliers are treated as a review signal rather than an automatic false-positive label.
- The final Member 4 score is a transparent screening score and should be interpreted separately from Member 2's predicted confirmed probability.
- Candidates with high screening scores should be prioritized for human review and follow-up analysis.
